In [ ]:
from matplotlib.pyplot import subplots, legend
from scipy.signal import sosfilt, sosfreqz
from scipy.fft import fft
from urllib.request import urlopen
from IPython.display import Audio
from findDigitalFilterIIRBiquadsByTargetFreq import findDigitalFilterIIRBiquadsByTargetFreq
import soundfile as sf
import pandas as pd
import numpy as np
import io

In [10]:
# Lê o arquivo de áudio 'record.wav' localizado na pasta '../assets/'
# 'data' é um array NumPy com os valores das amostras (pode ter múltiplos canais)
# 'samplerate' é o número de amostras por segundo (Hz), ex: 44100
data, samplerate = sf.read('../assets/record.wav')

# Reproduz o áudio do primeiro canal (canal esquerdo, se for estéreo)
# 'rate=samplerate' garante que a reprodução aconteça na velocidade correta
Audio(data[:, 0], rate=samplerate)

In [ ]:
# Calcula as seções biquad (SOS) de um filtro IIR Butterworth passa-baixa
# - fDesejada: frequência desejada em Hz (equivalente ao cutoff=0.3 normalizado, ou seja, 0.3 * fs/2)
# - ordem: ordem do filtro Butterworth (deve ser par)
# - fs=samplerate: frequência de amostragem do sinal original
# - filterType="lowpass": tipo do filtro. Aqui é passa-baixa (rejeita altas frequências)
sos, fc, f_comp = findDigitalFilterIIRBiquadsByTargetFreq(
    fDesejada=0.3 * samplerate / 2,
    ordem=8,
    fs=samplerate,
    filterType="lowpass",
    plot=False,
)

# Aplica o filtro ao sinal de áudio (somente no primeiro canal: data[:, 0])
# - sosfilt aplica o filtro IIR usando as seções biquad (SOS), mais estável numericamente
#   que aplicar os coeficientes de uma transfer function de alta ordem diretamente
filtered_data = sosfilt(sos, data[:, 0])

# Reproduz o sinal filtrado (após o filtro passa-baixa) com a mesma taxa de amostragem original
Audio(filtered_data, rate=samplerate)

In [ ]:
# Cria uma figura (fig) e um conjunto de eixos (ax) com tamanho 20x6 polegadas e cor de fundo cinza-claro
fig, ax = subplots(figsize=(20, 6), facecolor='#DEDEDE')

# Define a frequência máxima de visualização no gráfico em 10.000 Hz
# (poderia ser samplerate // 2, que é a frequência de Nyquist)
fmax = 10000
# Calcula o número de amostras que representam até 'fmax' Hz
# - np.size(data[:,0], axis=0): número total de amostras do canal 0 (mono ou canal esquerdo)
# - (total_amostras * fmax) // samplerate → número de amostras até fmax Hz
tsamples = (np.size(data[:, 0], axis=0) * fmax) // samplerate
# Cria o eixo X do gráfico: valores de frequência igualmente espaçados de 0 até fmax
x = np.linspace(0, fmax, tsamples)
# Calcula a FFT (transformada de Fourier) do sinal original e do filtrado
# [:tsamples] → limita a FFT à quantidade de amostras calculadas para até fmax Hz
y = fft(data[:, 0])[:tsamples]          # FFT do sinal original
yf = fft(filtered_data)[:tsamples]      # FFT do sinal filtrado

# Calcula a resposta em frequência do filtro IIR (biquads) criado
# - w: frequências
# - h: ganho complexo do filtro
# - worN=x: define as frequências nas quais a resposta será avaliada
# - fs=samplerate: taxa de amostragem do sinal (necessária para interpretar a resposta em Hz)
w, h = sosfreqz(sos, worN=x, fs=samplerate)
# Normaliza a altura do gráfico para escalar tudo entre 0 e 1
# Usa o maior valor (pico) entre os dois espectros de magnitude
ymax = max([max(abs(y)), max(abs(yf))])
# Plota os espectros:
# - linha vermelha: FFT do sinal original
# - linha azul: FFT do sinal filtrado
# - linha verde: resposta em frequência do filtro
line1, line2, line3 = ax.plot(
    np.real(x), abs(np.real(y / ymax)), "r",     # Sinal original normalizado
    np.real(x), abs(np.real(yf / ymax)), "b",    # Sinal filtrado normalizado
    np.real(x), abs(h), "g"                      # Resposta do filtro (não precisa normalizar)
)
# Adiciona legenda para as três curvas no gráfico
legend([line1, line2, line3], ["Normal", "Filtrado", "Filtro"])